<a href="https://colab.research.google.com/github/mafedcp65-netizen/Trabajo-de-grado/blob/main/Modelo_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SVM

In [1]:
!pip -q install openpyxl joblib scikit-learn pandas numpy

## Bloque 1. Cargar archivo de datos

In [2]:
from google.colab import files

uploaded = files.upload()

archivos_subidos = list(uploaded.keys())

ARCHIVO = next(
    (a for a in archivos_subidos if a.lower().endswith((".xlsx", ".xls", ".csv"))),
    None
)
ARCHIVO_MODELOS_ZIP = next(
    (a for a in archivos_subidos if a.lower().endswith(".zip")),
    None
)

if ARCHIVO is None:
    raise ValueError(
        "Debes subir la base de datos (.xlsx, .xls o .csv). "
        "Si la base no tiene etiquetas, puedes subir también el archivo .zip con los modelos."
    )

print(f"Base cargada: {ARCHIVO}")

if ARCHIVO_MODELOS_ZIP is not None:
    print(f"ZIP de modelos cargado: {ARCHIVO_MODELOS_ZIP}")
else:
    print("No se cargó ZIP de modelos en esta subida.")

Saving base etiquetada limpia.xlsx to base etiquetada limpia.xlsx
Base cargada: base etiquetada limpia.xlsx
No se cargó ZIP de modelos en esta subida.


## Bloque 2. Imports y configuración

In [4]:
import os
import json
import shutil
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

HOJA = "Base_estandarizada"
RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE_DENTRO_TRAIN = 0.20

COLUMNA_ETIQUETA = "Mención de cáncer"
COLUMNAS_TEXTO = [
    "Diagnóstico A",
    "Diagnóstico B",
    "Diagnóstico C",
    "Diagnóstico D",
    "Otros Estados Patológicos",
    "Otros Estados Patológicos 2",
]

PERCENTIL_BAJA_CONFIANZA_ETAPA2 = 0.20
UMBRAL_REASIGNACION_ETAPA1 = -0.10

ARCHIVO_SALIDA_CLASES = "predicciones_clase.xlsx"
ARCHIVO_SALIDA_SCORES = "predicciones_scores.xlsx"

MODO_EJECUCION = "auto"

RUTA_CARPETA_MODELOS = "modelos_svm_dos_etapas"

ARCHIVO_MODELOS_ZIP = globals().get("ARCHIVO_MODELOS_ZIP", None)

## Bloque 3. Funciones auxiliares

In [5]:
def limpiar_campo(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x in {"", "0", "0.0", "nan", "None"}:
        return ""
    return x.lower()

def construir_texto_fila(row):
    partes = []
    for col in COLUMNAS_TEXTO:
        valor = limpiar_campo(row[col])
        if valor:
            partes.append(valor)
    return " [SEP] ".join(partes)

def metricas_binarias(y_true, y_pred, pos_label=1):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, pos_label=pos_label, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, pos_label=pos_label, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, pos_label=pos_label, zero_division=0)),
    }

def metricas_multiclase(y_true, y_pred, labels=None):
    metricas = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

    if labels is not None:
        reporte_dict = classification_report(
            y_true, y_pred, labels=labels, output_dict=True, zero_division=0
        )
        for label in labels:
            clave = str(label)
            if clave in reporte_dict:
                metricas[f"precision_clase_{label}"] = float(reporte_dict[clave]["precision"])
                metricas[f"recall_clase_{label}"] = float(reporte_dict[clave]["recall"])
                metricas[f"f1_clase_{label}"] = float(reporte_dict[clave]["f1-score"])
    return metricas

def descripcion_score_etapa2(clases_modelo):
    if len(clases_modelo) != 2:
        return "Score no disponible"
    return (
        f"Score > 0 favorece clase {clases_modelo[1]}; "
        f"score < 0 favorece clase {clases_modelo[0]}"
    )

def detectar_etiquetas_validas(df):
    if COLUMNA_ETIQUETA not in df.columns:
        return False, None, None
    serie = pd.to_numeric(df[COLUMNA_ETIQUETA], errors="coerce")
    mask_validas = serie.isin([0, 1, 2])
    return bool(mask_validas.any()), serie, mask_validas

def evaluar_entrenabilidad(serie_etiquetas, mask_etiquetas_validas):
    if serie_etiquetas is None or mask_etiquetas_validas is None:
        return False, "La columna de etiqueta no existe o no contiene valores válidos 0, 1 o 2."

    y_valid = serie_etiquetas.loc[mask_etiquetas_validas].astype(int)
    n_validos = int(len(y_valid))

    if n_validos == 0:
        return False, "No hay filas con etiquetas válidas 0, 1 o 2."

    conteo_total = y_valid.value_counts().sort_index()

    if len(conteo_total) < 2:
        return False, (
            "La base no es entrenable: se necesitan al menos 2 clases distintas "
            f"en la etiqueta. Conteos actuales: {conteo_total.to_dict()}"
        )

    if (conteo_total < 2).any():
        return False, (
            "La base no es entrenable: cada clase presente necesita al menos 2 registros "
            f"para la partición estratificada. Conteos actuales: {conteo_total.to_dict()}"
        )

    y_etapa2 = y_valid[y_valid != 0]
    conteo_etapa2 = y_etapa2.value_counts().sort_index()

    if not {1, 2}.issubset(set(conteo_etapa2.index.tolist())):
        return False, (
            "La base no es entrenable para la etapa 2: se necesitan ejemplos de ambas "
            f"clases 1 y 2. Conteos actuales etapa 2: {conteo_etapa2.to_dict()}"
        )

    if (conteo_etapa2.reindex([1, 2], fill_value=0) < 2).any():
        return False, (
            "La base no es entrenable para la etapa 2: las clases 1 y 2 necesitan al menos "
            f"2 registros cada una. Conteos actuales etapa 2: {conteo_etapa2.to_dict()}"
        )

    return True, "La base tiene etiquetas suficientes para entrenar ambas etapas."

def cargar_base_datos(ruta_archivo, hoja_preferida=HOJA):
    ruta = Path(ruta_archivo)
    sufijo = ruta.suffix.lower()

    if sufijo == ".csv":
        df = pd.read_csv(ruta)
        return df, "csv"

    if sufijo not in {".xlsx", ".xls"}:
        raise ValueError(
            f"Formato no soportado: {sufijo}. Usa un archivo .xlsx, .xls o .csv."
        )

    try:
        df = pd.read_excel(ruta, sheet_name=hoja_preferida)
        return df, hoja_preferida
    except Exception:
        xls = pd.ExcelFile(ruta)
        hojas = xls.sheet_names

        if not hojas:
            raise ValueError("El archivo Excel no contiene hojas legibles.")

        for hoja in hojas:
            df_tmp = pd.read_excel(ruta, sheet_name=hoja)
            if all(col in df_tmp.columns for col in COLUMNAS_TEXTO):
                return df_tmp, hoja

        df = pd.read_excel(ruta, sheet_name=hojas[0])
        return df, hojas[0]

def guardar_modelos_en_carpeta(carpeta, modelo_etapa1, modelo_etapa2, metadata):
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)

    joblib.dump(modelo_etapa1, carpeta / "modelo_etapa1.joblib")
    joblib.dump(modelo_etapa2, carpeta / "modelo_etapa2.joblib")

    with open(carpeta / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"Modelos guardados en la carpeta: {carpeta.resolve()}")

def descomprimir_zip_modelos(ruta_zip, carpeta_destino):
    ruta_zip = Path(ruta_zip)
    carpeta_destino = Path(carpeta_destino)

    if not ruta_zip.exists():
        raise FileNotFoundError(f"No se encontró el ZIP de modelos: {ruta_zip}")

    if carpeta_destino.exists():
        shutil.rmtree(carpeta_destino)
    carpeta_destino.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        shutil.unpack_archive(str(ruta_zip), str(tmpdir))

        candidatos = [tmpdir] + [p for p in tmpdir.rglob("*") if p.is_dir()]
        carpeta_origen = None

        for candidato in candidatos:
            if (
                (candidato / "modelo_etapa1.joblib").exists()
                and (candidato / "modelo_etapa2.joblib").exists()
            ):
                carpeta_origen = candidato
                break

        if carpeta_origen is None:
            raise FileNotFoundError(
                "El ZIP no contiene los archivos esperados: "
                "'modelo_etapa1.joblib' y 'modelo_etapa2.joblib'."
            )

        for item in carpeta_origen.iterdir():
            destino = carpeta_destino / item.name
            if item.is_dir():
                shutil.copytree(item, destino, dirs_exist_ok=True)
            else:
                shutil.copy2(item, destino)

    print(f"ZIP de modelos descomprimido en: {carpeta_destino.resolve()}")

def preparar_y_cargar_modelos(ruta_carpeta, ruta_zip=None):
    carpeta = Path(ruta_carpeta)

    if ruta_zip is not None:
        descomprimir_zip_modelos(ruta_zip, carpeta)

    if not carpeta.exists() or not carpeta.is_dir():
        raise FileNotFoundError(
            f"No existe la carpeta de modelos: {carpeta}. "
            "Si la base no tiene etiquetas, sube también el ZIP de modelos en la misma carga o cuando el notebook lo pida."
        )

    ruta_etapa1 = carpeta / "modelo_etapa1.joblib"
    ruta_etapa2 = carpeta / "modelo_etapa2.joblib"
    ruta_metadata = carpeta / "metadata.json"

    if not ruta_etapa1.exists():
        raise FileNotFoundError(f"No se encontró: {ruta_etapa1}")
    if not ruta_etapa2.exists():
        raise FileNotFoundError(f"No se encontró: {ruta_etapa2}")

    modelo_etapa1 = joblib.load(ruta_etapa1)
    modelo_etapa2 = joblib.load(ruta_etapa2)

    metadata = {}
    if ruta_metadata.exists():
        with open(ruta_metadata, "r", encoding="utf-8") as f:
            metadata = json.load(f)

    return modelo_etapa1, modelo_etapa2, metadata

def pedir_zip_modelos_si_falta():
    global ARCHIVO_MODELOS_ZIP

    carpeta = Path(RUTA_CARPETA_MODELOS)
    carpeta_lista = (
        carpeta.exists()
        and (carpeta / "modelo_etapa1.joblib").exists()
        and (carpeta / "modelo_etapa2.joblib").exists()
    )

    if ARCHIVO_MODELOS_ZIP is None and not carpeta_lista:
        from google.colab import files
        print("La base no tiene etiquetas y no se encontró una carpeta local de modelos.")
        print("Sube ahora el archivo ZIP con los modelos entrenados.")
        uploaded_modelos = files.upload()
        zips_subidos = [a for a in uploaded_modelos.keys() if a.lower().endswith(".zip")]
        if not zips_subidos:
            raise ValueError(
                "No se subió ningún archivo .zip de modelos. "
                "Para una base sin etiquetas necesitas cargar el ZIP de modelos entrenados."
            )
        ARCHIVO_MODELOS_ZIP = zips_subidos[0]
        print(f"ZIP de modelos cargado en este paso: {ARCHIVO_MODELOS_ZIP}")


def evaluar_particion_modelo(nombre_particion, X_part, y_part, mejor_etapa1, mejor_etapa2, umbral_baja_confianza_etapa2):
    y_part = y_part.astype(int)

    y_part_etapa1 = (y_part != 0).astype(int)
    pred_etapa1_cruda = mejor_etapa1.predict(X_part)
    score_etapa1 = mejor_etapa1.decision_function(X_part)

    reasignado_de_0_a_2 = (
        (pred_etapa1_cruda == 0) &
        (score_etapa1 > UMBRAL_REASIGNACION_ETAPA1)
    )
    pred_etapa1_binaria = np.where(reasignado_de_0_a_2, 1, pred_etapa1_cruda).astype(int)

    metricas_etapa1 = metricas_binarias(y_part_etapa1, pred_etapa1_binaria, pos_label=1)
    cm_etapa1 = confusion_matrix(y_part_etapa1, pred_etapa1_binaria)
    reporte_etapa1 = classification_report(y_part_etapa1, pred_etapa1_binaria, digits=4, zero_division=0)

    mask_pos = y_part != 0
    X_part_pos = X_part[mask_pos]
    y_part_pos = y_part[mask_pos]

    if len(y_part_pos) > 0:
        pred_etapa2_pos = mejor_etapa2.predict(X_part_pos)
        metricas_etapa2 = metricas_multiclase(y_part_pos, pred_etapa2_pos, labels=[1, 2])
        cm_etapa2 = confusion_matrix(y_part_pos, pred_etapa2_pos, labels=[1, 2])
        reporte_etapa2 = classification_report(y_part_pos, pred_etapa2_pos, labels=[1, 2], digits=4, zero_division=0)
    else:
        pred_etapa2_pos = np.array([])
        metricas_etapa2 = None
        cm_etapa2 = None
        reporte_etapa2 = "Sin casos positivos reales en esta partición."

    pred_etapa2_todo = mejor_etapa2.predict(X_part)
    score_etapa2_todo = mejor_etapa2.decision_function(X_part)

    pred_final = np.where(pred_etapa1_binaria == 0, 0, pred_etapa2_todo).astype(int)
    pred_final[reasignado_de_0_a_2] = 2

    bandera_baja_confianza_etapa2 = (
        (pred_etapa1_binaria == 1) &
        (~reasignado_de_0_a_2) &
        (np.abs(score_etapa2_todo) <= umbral_baja_confianza_etapa2)
    )
    bandera_revision_manual = (pred_final == 2) | bandera_baja_confianza_etapa2

    metricas_finales = metricas_multiclase(y_part, pred_final, labels=[0, 1, 2])
    cm_final = confusion_matrix(y_part, pred_final, labels=[0, 1, 2])
    reporte_final = classification_report(y_part, pred_final, labels=[0, 1, 2], digits=4, zero_division=0)

    return {
        "nombre": nombre_particion,
        "n": int(len(X_part)),
        "distribucion_clases": {
            str(k): int(v) for k, v in y_part.value_counts().sort_index().items()
        },
        "etapa_1": {
            "metricas": metricas_etapa1,
            "matriz_confusion": cm_etapa1,
            "reporte": reporte_etapa1,
            "casos_reasignados_0_a_2": int(reasignado_de_0_a_2.sum()),
        },
        "etapa_2": {
            "metricas": metricas_etapa2,
            "matriz_confusion": cm_etapa2,
            "reporte": reporte_etapa2,
            "n_positivos_reales": int(mask_pos.sum()),
        },
        "modelo_final": {
            "metricas": metricas_finales,
            "matriz_confusion": cm_final,
            "reporte": reporte_final,
            "casos_marcados_revision_manual": int(bandera_revision_manual.sum()),
        },
    }

## Bloque 4. Carga, preparación de datos y detección del modo

In [6]:
df_base, hoja_detectada = cargar_base_datos(ARCHIVO, HOJA)

column_rename_map = {
    "otros estados patologicos": "Otros Estados Patológicos",
    "otros estados patologicos 2": "Otros Estados Patológicos 2",
}

df_base.columns = [
    column_rename_map.get(col.lower().replace('ó', 'o'), col)
    for col in df_base.columns
]

faltantes_texto = [c for c in COLUMNAS_TEXTO if c not in df_base.columns]
if faltantes_texto:
    raise ValueError(
        f"Faltan columnas requeridas en la base cargada ({hoja_detectada}): {faltantes_texto}"
    )

filas_iniciales = len(df_base)

df_base["texto"] = df_base.apply(construir_texto_fila, axis=1)
df_base = df_base[df_base["texto"].str.strip() != ""].copy()

tiene_etiquetas_validas, serie_etiquetas, mask_etiquetas_validas = detectar_etiquetas_validas(df_base)
base_entrenable, motivo_entrenabilidad = evaluar_entrenabilidad(
    serie_etiquetas, mask_etiquetas_validas
)

if MODO_EJECUCION.lower() == "auto":
    MODO_REAL = "train" if base_entrenable else "inferencia"
elif MODO_EJECUCION.lower() == "train":
    if not base_entrenable:
        raise ValueError(
            "Se solicitó entrenamiento, pero la base no es entrenable. "
            + motivo_entrenabilidad
        )
    MODO_REAL = "train"
elif MODO_EJECUCION.lower() == "inferencia":
    MODO_REAL = "inferencia"
else:
    raise ValueError("MODO_EJECUCION debe ser 'auto', 'train' o 'inferencia'.")

if MODO_REAL == "train":
    df = df_base.loc[mask_etiquetas_validas].copy()
    df[COLUMNA_ETIQUETA] = serie_etiquetas.loc[mask_etiquetas_validas].astype(int)

    print("Modo seleccionado: ENTRENAMIENTO")
    print("Origen cargado:", hoja_detectada)
    print("Filas iniciales:", filas_iniciales)
    print("Filas utilizables con texto:", len(df_base))
    print("Filas utilizables con etiqueta válida:", len(df))
    print("Diagnóstico de entrenabilidad:", motivo_entrenabilidad)
    print("\nDistribución de clases:")
    print(df[COLUMNA_ETIQUETA].value_counts().sort_index())
else:
    df = df_base.copy()
    print("Modo seleccionado: INFERENCIA")
    print("Origen cargado:", hoja_detectada)
    print("Filas iniciales:", filas_iniciales)
    print("Filas utilizables con texto:", len(df))
    print("Motivo para no entrenar:", motivo_entrenabilidad)
    if ARCHIVO_MODELOS_ZIP is not None:
        print(f"Se usará el ZIP de modelos cargado: {ARCHIVO_MODELOS_ZIP}")
    else:
        print(
            "No se entrenará con esta base. "
            "Se usará la carpeta de modelos local si existe; si no, se te pedirá el ZIP."
        )

Modo seleccionado: ENTRENAMIENTO
Origen cargado: Base_estandarizada
Filas iniciales: 4000
Filas utilizables con texto: 4000
Filas utilizables con etiqueta válida: 3999
Diagnóstico de entrenabilidad: La base tiene etiquetas suficientes para entrenar ambas etapas.

Distribución de clases:
Mención de cáncer
0    3052
1     908
2      39
Name: count, dtype: int64


## Bloque 5. Ejecución principal

In [7]:
predicciones_clase = None
predicciones_scores = None
pred_final = None
reasignado_de_0_a_2 = None

metricas_etapa1 = cm_etapa1 = reporte_etapa1 = None
metricas_etapa2 = cm_etapa2 = reporte_etapa2 = None
metricas_finales = cm_final = reporte_final = None
bandera_revision_manual = None
umbral_baja_confianza_etapa2 = None
metadata_modelos = {}

grid_etapa1 = grid_etapa2 = None
mejor_etapa1 = mejor_etapa2 = None

X_train = X_val = X_test = None
y_train = y_val = y_test = None
X_train_val = y_train_val = None
idx_train = idx_val = idx_test = idx_train_val = None

metricas_particiones = {}

if MODO_REAL == "train":
    X = df["texto"]
    y = df[COLUMNA_ETIQUETA]

    conteo_clases = y.value_counts()
    clases_presentes = sorted(conteo_clases.index.tolist())
    if len(clases_presentes) < 2:
        raise ValueError(
            "La base etiquetada no tiene suficientes clases para entrenar. "
            f"Clases encontradas: {clases_presentes}"
        )
    if (conteo_clases < 2).any():
        raise ValueError(
            "Cada clase necesita al menos 2 registros para poder hacer partición estratificada. "
            f"Conteos actuales: {conteo_clases.to_dict()}"
        )

    X_train_val, X_test, y_train_val, y_test, idx_train_val, idx_test = train_test_split(
        X, y, df.index,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )

    X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
        X_train_val, y_train_val, idx_train_val,
        test_size=VAL_SIZE_DENTRO_TRAIN,
        random_state=RANDOM_STATE,
        stratify=y_train_val
    )

    print("Tamaño train:", len(X_train))
    print("Tamaño val:", len(X_val))
    print("Tamaño test:", len(X_test))

    y_train_etapa1 = (y_train != 0).astype(int)

    pipeline_etapa1 = Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, strip_accents="unicode")),
        ("svm", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, max_iter=5000)),
    ])

    param_grid_etapa1 = [
        {
            "tfidf__analyzer": ["char_wb"],
            "tfidf__ngram_range": [(3, 5), (4, 6)],
            "tfidf__min_df": [2, 5],
            "tfidf__sublinear_tf": [True],
            "svm__C": [0.25, 0.5, 1.0, 2.0, 4.0],
        },
        {
            "tfidf__analyzer": ["word"],
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__min_df": [1, 2, 5],
            "tfidf__max_df": [0.95, 0.98],
            "tfidf__sublinear_tf": [True],
            "svm__C": [0.25, 0.5, 1.0, 2.0, 4.0],
        },
    ]

    cv_etapa1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    grid_etapa1 = GridSearchCV(
        estimator=pipeline_etapa1,
        param_grid=param_grid_etapa1,
        scoring="recall",
        cv=cv_etapa1,
        n_jobs=-1,
        verbose=1,
        refit=True,
        return_train_score=False,
    )

    grid_etapa1.fit(X_train, y_train_etapa1)
    mejor_etapa1 = grid_etapa1.best_estimator_

    mask_train_pos = y_train != 0
    X_train_pos = X_train[mask_train_pos]
    y_train_pos = y_train[mask_train_pos]

    if len(np.unique(y_train_pos)) < 2:
        raise ValueError(
            "La etapa 2 necesita ejemplos de ambas clases (1 y 2) en train para entrenar."
        )

    pipeline_etapa2 = Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, strip_accents="unicode")),
        ("svm", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, max_iter=5000)),
    ])

    param_grid_etapa2 = [
        {
            "tfidf__analyzer": ["word"],
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__min_df": [1, 2],
            "tfidf__max_df": [0.95, 0.98],
            "tfidf__sublinear_tf": [True],
            "svm__C": [0.25, 0.5, 1.0, 2.0, 4.0],
        },
        {
            "tfidf__analyzer": ["char_wb"],
            "tfidf__ngram_range": [(3, 5), (4, 6)],
            "tfidf__min_df": [1, 2],
            "tfidf__sublinear_tf": [True],
            "svm__C": [0.25, 0.5, 1.0, 2.0],
        },
    ]

    cv_etapa2 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    grid_etapa2 = GridSearchCV(
        estimator=pipeline_etapa2,
        param_grid=param_grid_etapa2,
        scoring="recall_macro",
        cv=cv_etapa2,
        n_jobs=-1,
        verbose=1,
        refit=True,
        return_train_score=False,
    )

    grid_etapa2.fit(X_train_pos, y_train_pos)
    mejor_etapa2 = grid_etapa2.best_estimator_

    score_etapa2_train = mejor_etapa2.decision_function(X_train_pos)
    umbral_baja_confianza_etapa2 = float(
        np.quantile(np.abs(score_etapa2_train), PERCENTIL_BAJA_CONFIANZA_ETAPA2)
    )

    metricas_particiones = {
        "train": evaluar_particion_modelo("train", X_train, y_train, mejor_etapa1, mejor_etapa2, umbral_baja_confianza_etapa2),
        "val": evaluar_particion_modelo("val", X_val, y_val, mejor_etapa1, mejor_etapa2, umbral_baja_confianza_etapa2),
        "test": evaluar_particion_modelo("test", X_test, y_test, mejor_etapa1, mejor_etapa2, umbral_baja_confianza_etapa2),
    }

    metricas_etapa1 = metricas_particiones["test"]["etapa_1"]["metricas"]
    cm_etapa1 = metricas_particiones["test"]["etapa_1"]["matriz_confusion"]
    reporte_etapa1 = metricas_particiones["test"]["etapa_1"]["reporte"]

    metricas_etapa2 = metricas_particiones["test"]["etapa_2"]["metricas"]
    cm_etapa2 = metricas_particiones["test"]["etapa_2"]["matriz_confusion"]
    reporte_etapa2 = metricas_particiones["test"]["etapa_2"]["reporte"]

    metricas_finales = metricas_particiones["test"]["modelo_final"]["metricas"]
    cm_final = metricas_particiones["test"]["modelo_final"]["matriz_confusion"]
    reporte_final = metricas_particiones["test"]["modelo_final"]["reporte"]

    metadata_modelos = {
        "hoja": HOJA,
        "columnas_texto": COLUMNAS_TEXTO,
        "columna_etiqueta": COLUMNA_ETIQUETA,
        "percentil_baja_confianza_etapa2": PERCENTIL_BAJA_CONFIANZA_ETAPA2,
        "umbral_baja_confianza_etapa2": umbral_baja_confianza_etapa2,
        "umbral_reasignacion_etapa1": UMBRAL_REASIGNACION_ETAPA1,
        "mejores_hiperparametros_etapa1": grid_etapa1.best_params_,
        "mejores_hiperparametros_etapa2": grid_etapa2.best_params_,
        "clases_modelo_etapa2": mejor_etapa2.named_steps["svm"].classes_.tolist(),
        "interpretacion_score_etapa2": descripcion_score_etapa2(mejor_etapa2.named_steps["svm"].classes_),
    }

    guardar_modelos_en_carpeta(
        RUTA_CARPETA_MODELOS,
        mejor_etapa1,
        mejor_etapa2,
        metadata_modelos
    )

    pred_etapa1_test_cruda = mejor_etapa1.predict(X_test)
    score_etapa1_test = mejor_etapa1.decision_function(X_test)

    reasignado_de_0_a_2 = (
        (pred_etapa1_test_cruda == 0) &
        (score_etapa1_test > UMBRAL_REASIGNACION_ETAPA1)
    )
    pred_etapa1_test_binaria = np.where(reasignado_de_0_a_2, 1, pred_etapa1_test_cruda).astype(int)

    pred_etapa2_todo = mejor_etapa2.predict(X_test)
    score_etapa2_todo = mejor_etapa2.decision_function(X_test)

    pred_final = np.where(pred_etapa1_test_binaria == 0, 0, pred_etapa2_todo).astype(int)
    pred_final[reasignado_de_0_a_2] = 2

    bandera_baja_confianza_etapa2 = (
        (pred_etapa1_test_binaria == 1) &
        (~reasignado_de_0_a_2) &
        (np.abs(score_etapa2_todo) <= umbral_baja_confianza_etapa2)
    )

    bandera_revision_manual = (pred_final == 2) | bandera_baja_confianza_etapa2

    salida_base = df.loc[idx_test].copy()
    salida_base["texto"] = X_test.values

    predicciones_clase = pd.DataFrame({
        "id_caso": salida_base["ID_PSEUDO"],
        "texto": salida_base["texto"].values,
        "pred_etapa1_cruda": pred_etapa1_test_cruda,
        "reasignado_de_0_a_2": reasignado_de_0_a_2,
        "clase_predicha": pred_final,
    })

    predicciones_scores = pd.DataFrame({
        "id_caso": salida_base["ID_PSEUDO"],
        "texto": salida_base["texto"].values,
        "score_etapa1_hacia_mencion": score_etapa1_test,
        "score_etapa2_hacia_clase_2": score_etapa2_todo,
        "abs_score_etapa2": np.abs(score_etapa2_todo),
        "reasignado_de_0_a_2": reasignado_de_0_a_2,
    })

else:
    pedir_zip_modelos_si_falta()

    mejor_etapa1, mejor_etapa2, metadata_modelos = preparar_y_cargar_modelos(
        RUTA_CARPETA_MODELOS,
        ARCHIVO_MODELOS_ZIP,
    )

    X_inferencia = df["texto"]

    pred_etapa1_inf_cruda = mejor_etapa1.predict(X_inferencia)
    score_etapa1_inf = mejor_etapa1.decision_function(X_inferencia)

    umbral_reasignacion_etapa1 = metadata_modelos.get(
        "umbral_reasignacion_etapa1",
        UMBRAL_REASIGNACION_ETAPA1,
    )

    reasignado_de_0_a_2 = (
        (pred_etapa1_inf_cruda == 0) &
        (score_etapa1_inf > umbral_reasignacion_etapa1)
    )
    pred_etapa1_inf_binaria = np.where(reasignado_de_0_a_2, 1, pred_etapa1_inf_cruda).astype(int)

    pred_etapa2_inf = mejor_etapa2.predict(X_inferencia)
    pred_final = np.where(pred_etapa1_inf_binaria == 0, 0, pred_etapa2_inf).astype(int)
    pred_final[reasignado_de_0_a_2] = 2

    predicciones_clase = pd.DataFrame({
        "id_caso": df["ID_PSEUDO"],
        "texto": X_inferencia.values,
        "pred_etapa1_cruda": pred_etapa1_inf_cruda,
        "reasignado_de_0_a_2": reasignado_de_0_a_2,
        "clase_predicha": pred_final,
    })

    score_etapa2_inf = mejor_etapa2.decision_function(X_inferencia)

    umbral_baja_confianza_etapa2 = metadata_modelos.get("umbral_baja_confianza_etapa2", 0.0)

    bandera_baja_confianza_etapa2 = (
        (pred_etapa1_inf_binaria == 1) &
        (~reasignado_de_0_a_2) &
        (np.abs(score_etapa2_inf) <= umbral_baja_confianza_etapa2)
    )

    bandera_revision_manual = (pred_final == 2) | bandera_baja_confianza_etapa2

    predicciones_scores = pd.DataFrame({
        "id_caso": df["ID_PSEUDO"],
        "texto": X_inferencia.values,
        "score_etapa1_hacia_mencion": score_etapa1_inf,
        "score_etapa2_hacia_clase_2": score_etapa2_inf,
        "abs_score_etapa2": np.abs(score_etapa2_inf),
        "reasignado_de_0_a_2": reasignado_de_0_a_2,
    })

Tamaño train: 2559
Tamaño val: 640
Tamaño test: 800
Fitting 5 folds for each of 80 candidates, totalling 400 fits
Fitting 3 folds for each of 56 candidates, totalling 168 fits
Modelos guardados en la carpeta: /content/modelos_svm_dos_etapas


## Bloque 6. Resumen completo de métricas

In [8]:
if MODO_REAL == "train":
    resumen_metricas = {
        "filas_iniciales": int(filas_iniciales),
        "filas_utilizables_texto": int(len(df_base)),
        "filas_utilizables_etiqueta": int(len(df)),
        "tamano_train": int(len(X_train)),
        "tamano_val": int(len(X_val)),
        "tamano_test": int(len(X_test)),
        "distribucion_clases_total": {
            str(k): int(v) for k, v in df[COLUMNA_ETIQUETA].value_counts().sort_index().items()
        },
        "etapa_1": {
            "criterio_cv": "recall",
            "mejor_score_cv": float(grid_etapa1.best_score_),
            "mejores_hiperparametros": grid_etapa1.best_params_,
            "umbral_reasignacion_etapa1": UMBRAL_REASIGNACION_ETAPA1,
        },
        "etapa_2": {
            "criterio_cv": "recall_macro",
            "mejor_score_cv": float(grid_etapa2.best_score_),
            "mejores_hiperparametros": grid_etapa2.best_params_,
            "umbral_baja_confianza_etapa2": umbral_baja_confianza_etapa2,
            "clases_modelo": mejor_etapa2.named_steps["svm"].classes_.tolist(),
            "interpretacion_score": descripcion_score_etapa2(mejor_etapa2.named_steps["svm"].classes_),
        },
        "particiones": {
            nombre: {
                "n": datos["n"],
                "distribucion_clases": datos["distribucion_clases"],
                "etapa_1": {
                    "metricas": datos["etapa_1"]["metricas"],
                    "matriz_confusion": datos["etapa_1"]["matriz_confusion"].tolist(),
                },
                "etapa_2": {
                    "metricas": datos["etapa_2"]["metricas"],
                    "matriz_confusion": None if datos["etapa_2"]["matriz_confusion"] is None else datos["etapa_2"]["matriz_confusion"].tolist(),
                },
                "modelo_final": {
                    "metricas": datos["modelo_final"]["metricas"],
                    "matriz_confusion": datos["modelo_final"]["matriz_confusion"].tolist(),
                },
            }
            for nombre, datos in metricas_particiones.items()
        }
    }

    print("="*80)
    print("RESUMEN DE MÉTRICAS")
    print("="*80)
    print("\nPartición usada:")
    print("- Test: 20% del total")
    print("- Validación: 20% del 80% restante")
    print("- Train final: 64% del total")
    print(f"\nTamaños -> train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")

    print("\nETAPA 1: 0 vs (1+2)")
    print("Mejores hiperparámetros:", grid_etapa1.best_params_)
    print(f"Mejor recall CV: {grid_etapa1.best_score_:.4f}")
    print(f"Umbral de reasignación etapa 1: {UMBRAL_REASIGNACION_ETAPA1:.6f}")

    print("\nETAPA 2: 1 vs 2")
    print("Mejores hiperparámetros:", grid_etapa2.best_params_)
    print(f"Mejor recall macro CV: {grid_etapa2.best_score_:.4f}")
    print("Interpretación del score:", descripcion_score_etapa2(mejor_etapa2.named_steps["svm"].classes_))
    print(f"Umbral de baja confianza etapa 2: {umbral_baja_confianza_etapa2:.6f}")

    for nombre in ["train", "val", "test"]:
        datos = metricas_particiones[nombre]

        print("\n" + "="*80)
        print(f"PARTICIÓN: {nombre.upper()}")
        print(f"Tamaño: {datos['n']}")
        print("Distribución de clases:", datos["distribucion_clases"])

        print("\nETAPA 1")
        print("Casos reasignados de 0 a 2:", datos["etapa_1"]["casos_reasignados_0_a_2"])
        print("Métricas:", datos["etapa_1"]["metricas"])
        print("Matriz de confusión:")
        print(datos["etapa_1"]["matriz_confusion"])
        print("Reporte:")
        print(datos["etapa_1"]["reporte"])

        print("\nETAPA 2")
        print("Métricas:", datos["etapa_2"]["metricas"])
        print("Matriz de confusión:")
        print(datos["etapa_2"]["matriz_confusion"])
        print("Reporte:")
        print(datos["etapa_2"]["reporte"])

        print("\nMODELO FINAL JERÁRQUICO")
        print("Métricas:", datos["modelo_final"]["metricas"])
        print("Matriz de confusión:")
        print(datos["modelo_final"]["matriz_confusion"])
        print("Reporte:")
        print(datos["modelo_final"]["reporte"])

else:
    print("Modo inferencia: no se calculan métricas ni scores.")
    print("Metadata cargada de la carpeta de modelos:")
    print(metadata_modelos if metadata_modelos else "Sin metadata.json")

RESUMEN DE MÉTRICAS

Partición usada:
- Test: 20% del total
- Validación: 20% del 80% restante
- Train final: 64% del total

Tamaños -> train: 2559, val: 640, test: 800

ETAPA 1: 0 vs (1+2)
Mejores hiperparámetros: {'svm__C': 4.0, 'tfidf__analyzer': 'char_wb', 'tfidf__min_df': 2, 'tfidf__ngram_range': (3, 5), 'tfidf__sublinear_tf': True}
Mejor recall CV: 0.9785
Umbral de reasignación etapa 1: -0.100000

ETAPA 2: 1 vs 2
Mejores hiperparámetros: {'svm__C': 0.25, 'tfidf__analyzer': 'char_wb', 'tfidf__min_df': 2, 'tfidf__ngram_range': (4, 6), 'tfidf__sublinear_tf': True}
Mejor recall macro CV: 0.7423
Interpretación del score: Score > 0 favorece clase 2; score < 0 favorece clase 1
Umbral de baja confianza etapa 2: 0.565496

PARTICIÓN: TRAIN
Tamaño: 2559
Distribución de clases: {'0': 1953, '1': 581, '2': 25}

ETAPA 1
Casos reasignados de 0 a 2: 0
Métricas: {'accuracy': 0.9996092223524814, 'precision': 1.0, 'recall': 0.9983498349834984, 'f1': 0.9991742361684558}
Matriz de confusión:
[[1953   

In [9]:
import pandas as pd

df_para_svm = pd.DataFrame({
    "id_caso": df.loc[idx_test, "ID_PSEUDO"].values,
    "clase_real": y_test.values,
    "clase_predicha": pred_final
})

df_para_svm.rename(columns={
    "id_caso": "id",
    "clase_real": "y_true",
    "clase_predicha": "y_pred"
}, inplace=True)

df_para_svm.dropna(subset=["y_true", "y_pred"], inplace=True)

ARCHIVO_PARA_SVM = "Para svm.xlsx"

df_para_svm.to_excel(ARCHIVO_PARA_SVM, index=False)
print(f"Archivo generado: {ARCHIVO_PARA_SVM}")

from google.colab import files
files.download(ARCHIVO_PARA_SVM)


Archivo generado: Para svm.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Bloque 7. Guardar salidas

In [10]:
predicciones_clase.to_excel(ARCHIVO_SALIDA_CLASES, index=False)
print("Archivo generado:")
print("-", ARCHIVO_SALIDA_CLASES)

if MODO_REAL == "train" and predicciones_scores is not None:
    predicciones_scores.to_excel(ARCHIVO_SALIDA_SCORES, index=False)
    print("-", ARCHIVO_SALIDA_SCORES)
else:
    print("No se genera archivo de scores en modo inferencia.")

Archivo generado:
- predicciones_clase.xlsx
- predicciones_scores.xlsx


## Bloque 8. Descargas

In [11]:
from google.colab import files

files.download(ARCHIVO_SALIDA_CLASES)

if MODO_REAL == "train" and predicciones_scores is not None:
    files.download(ARCHIVO_SALIDA_SCORES)

    nombre_zip_modelos = shutil.make_archive(
        base_name=str(Path(RUTA_CARPETA_MODELOS)),
        format="zip",
        root_dir=str(Path(RUTA_CARPETA_MODELOS).parent),
        base_dir=str(Path(RUTA_CARPETA_MODELOS).name),
    )
    print(f"También se creó el archivo: {nombre_zip_modelos}")
    files.download(nombre_zip_modelos)
else:
    print("En modo inferencia solo se descarga el Excel de clases.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

También se creó el archivo: /content/modelos_svm_dos_etapas.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpretación de los scores y reglas del modelo SVM

### Score etapa 1 (`score_etapa1_hacia_mencion`)
Corresponde a la clasificación entre **sin mención de cáncer (0)** y **con mención de cáncer (1 o 2)**.

- **Score > 0**: el caso se inclina hacia **mención de cáncer**
- **Score < 0**: el caso se inclina hacia **sin mención de cáncer**
- **Score cercano a 0**: el modelo tiene **menor seguridad**
- **Score alejado de 0**: el modelo tiene **mayor seguridad**

### Predicción cruda etapa 1 (`pred_etapa1_cruda`)
Es la salida original de la etapa 1 antes de aplicar la regla de seguridad.

- **0**: el modelo predice **sin mención de cáncer**
- **1**: el modelo predice **con mención de cáncer**

### Reasignación de 0 a 2 (`reasignado_de_0_a_2`)
Corresponde a la regla operativa de seguridad aplicada en la etapa 1.

- Si un caso fue clasificado inicialmente como **0**
- pero su `score_etapa1_hacia_mencion` quedó **cerca de la frontera de decisión**
- entonces el caso se **reasigna directamente a clase 2**

Esto se hace para **reducir el riesgo de falsos 0**, es decir, evitar que un caso con posible mención de cáncer quede descartado como negativo.

- **True**: el caso fue reasignado de **0 a 2**
- **False**: el caso siguió el flujo normal del modelo

### Score etapa 2 (`score_etapa2_hacia_clase_2`)
Corresponde a la clasificación entre **clase 1** y **clase 2**, solo para los casos que pasaron como positivos en la etapa 1.

- **Score > 0**: el caso se inclina hacia **clase 2**
- **Score < 0**: el caso se inclina hacia **clase 1**
- **Score cercano a 0**: existe **duda entre ambas clases**
- **Score alejado de 0**: existe **mayor separación entre clases**

### Valor absoluto del score etapa 2 (`abs_score_etapa2`)
Es el valor absoluto de `score_etapa2_hacia_clase_2`.

- **Valor bajo**: el caso está **cerca de la frontera de decisión**, por lo que genera **menor confianza**
- **Valor alto**: el caso está **más lejos de la frontera**, por lo que genera **mayor confianza**

### Predicción final del modelo (`pred_svm_final`)
Es la salida final del pipeline SVM después de aplicar la lógica jerárquica y la regla de seguridad.

- **0**: sin mención de cáncer
- **1**: mención clara de cáncer
- **2**: mención probable o sospechosa de cáncer

### Nota importante
Estos scores **no son probabilidades**. Indican la **dirección de la predicción** y la **distancia respecto a la frontera de decisión** del modelo.

Además, la reasignación de algunos casos de **0 a 2** **no cambia el entrenamiento del SVM**, sino la **regla operativa de decisión del pipeline**, con el fin de priorizar que los casos dudosos no queden clasificados como 0.

###Informacion sobre los que fallan

In [12]:
import pandas as pd
import numpy as np
from IPython.display import display
from google.colab.data_table import DataTable

df_resultados_completos = pd.DataFrame()

if MODO_REAL == "train":

    X_base = X_train
    y_base = y_train

    df_base = pd.DataFrame({
        "ID_PSEUDO": df.loc[X_train.index, "ID_PSEUDO"] if hasattr(X_train, "index") else df["ID_PSEUDO"].iloc[:len(X_base)].values,
        "texto": X_base,
        COLUMNA_ETIQUETA: y_base
    }).reset_index(drop=True)

    X_base = df_base["texto"]
    y_base = df_base[COLUMNA_ETIQUETA]


    pred_etapa1_cruda_all = mejor_etapa1.predict(X_base)
    score_etapa1_all = mejor_etapa1.decision_function(X_base)


    reasignado_de_0_a_2_all = (
        (pred_etapa1_cruda_all == 0) &
        (score_etapa1_all > UMBRAL_REASIGNACION_ETAPA1)
    )
    pred_etapa1_binaria_all = np.where(
        reasignado_de_0_a_2_all, 1, pred_etapa1_cruda_all
    ).astype(int)


    pred_etapa2_all = mejor_etapa2.predict(X_base)
    score_etapa2_all = mejor_etapa2.decision_function(X_base)
    abs_score_etapa2_all = np.abs(score_etapa2_all)


    pred_final_all = np.where(pred_etapa1_binaria_all == 0, 0, pred_etapa2_all).astype(int)
    pred_final_all[reasignado_de_0_a_2_all] = 2

    df_resultados_completos = pd.DataFrame({
        "id_caso": df_base["ID_PSEUDO"].values,
        "clase_predicha": pred_final_all,
        "clase_real": y_base.values,
        "score_etapa1_hacia_mencion": score_etapa1_all,
        "score_etapa2_hacia_clase_2": score_etapa2_all,
        "abs_score_etapa2": abs_score_etapa2_all,
        "pred_etapa1_cruda": pred_etapa1_cruda_all,
        "reasignado_de_0_a_2": reasignado_de_0_a_2_all,
        "texto": X_base.values
    })


    df_resultados_completos = df_resultados_completos[
        df_resultados_completos["clase_real"].notna() &
        df_resultados_completos["clase_predicha"].notna() &
        (df_resultados_completos["clase_predicha"] != df_resultados_completos["clase_real"])
    ].copy()

else:
    df_resultados_completos = predicciones_clase.merge(
        predicciones_scores,
        on=["id_caso", "texto", "reasignado_de_0_a_2", "pred_etapa1_cruda"],
        how="left"
    )

    if COLUMNA_ETIQUETA in df.columns:
        df_resultados_completos = df_resultados_completos.merge(
            df[["ID_PSEUDO", COLUMNA_ETIQUETA]].rename(
                columns={"ID_PSEUDO": "id_caso", COLUMNA_ETIQUETA: "clase_real"}
            ),
            on="id_caso",
            how="left"
        )
    else:
        df_resultados_completos["clase_real"] = pd.NA

df_resultados_completos = df_resultados_completos[[
    "id_caso",
    "clase_predicha",
    "clase_real",
    "score_etapa1_hacia_mencion",
    "score_etapa2_hacia_clase_2",
    "abs_score_etapa2",
    "pred_etapa1_cruda",
    "reasignado_de_0_a_2",
    "texto"
]].copy()

print("Cantidad de errores encontrados:", len(df_resultados_completos))

display(df_resultados_completos)

display(DataTable(df_resultados_completos, include_index=False, num_rows_per_page=10))

print("\n--- Definición de Columnas ---")
print("id_caso: Identificador único del caso.")
print("clase_predicha: Clase predicha por el modelo final (0: sin mención de cáncer, 1: mención clara, 2: mención probable/sospechosa).")
print("clase_real: Clase real o etiquetada del caso (si disponible).")
print("score_etapa1_hacia_mencion: Score de la primera etapa del modelo (0 vs 1+2). Positivo indica mención, negativo sin mención.")
print("score_etapa2_hacia_clase_2: Score de la segunda etapa del modelo (1 vs 2). Positivo indica clase 2, negativo clase 1.")
print("abs_score_etapa2: Valor absoluto del score de la segunda etapa. Valores bajos indican baja confianza.")
print("pred_etapa1_cruda: Predicción original de la primera etapa antes de cualquier reasignación (0 o 1).")
print("reasignado_de_0_a_2: Booleano que indica si el caso fue reasignado de clase 0 a clase 2 debido a una baja confianza en la etapa 1.")
print("texto: El texto combinado utilizado para la clasificación.")

ARCHIVO_RESULTADOS_COMPLETOS = "resultados_train_svm_solo_errores.xlsx"
df_resultados_completos.to_excel(ARCHIVO_RESULTADOS_COMPLETOS, index=False)
print(f"Archivo generado: {ARCHIVO_RESULTADOS_COMPLETOS}")

from google.colab import files
files.download(ARCHIVO_RESULTADOS_COMPLETOS)


Cantidad de errores encontrados: 12


,id_caso,clase_predicha,clase_real,score_etapa1_hacia_mencion,score_etapa2_hacia_clase_2,abs_score_etapa2,pred_etapa1_cruda,reasignado_de_0_a_2,texto
1,CASO_002951,2,1,2.213094,0.612212,0.612212,1,False,tumor d comportamiento incierto o deconocido d...
86,CASO_002865,2,1,1.616272,0.104461,0.104461,1,False,neoplasia abdominal maligna
398,CASO_002511,2,1,0.984561,0.336357,0.336357,1,False,disfunci”n organica [SEP] sepsis de origen abd...
476,CASO_002202,2,1,1.337647,0.266994,0.266994,1,False,tumor maligno de las vias biliares [SEP] diabe...
704,CASO_000990,2,1,1.006016,0.172760,0.172760,1,False,enfermedad cerebrovascular isquemica [SEP] hip...
1028,CASO_003363,0,1,-0.637667,-0.315092,0.315092,0,False,choque cardiogenico [SEP] infarto agudo del mi...
1412,CASO_003261,2,1,1.535257,0.381371,0.381371,1,False,hipoxemia refractaria [SEP] síndrome de distre...
1441,CASO_003369,2,1,1.088699,0.097150,0.097150,1,False,cáncer pulmonar [SEP] tabaquismo [SEP] enferme...
1485,CASO_001695,2,1,1.133257,0.333004,0.333004,1,False,paro cardiaco [SEP] tumor de comportamiento in...
1966,CASO_000445,2,1,0.921130,0.018855,0.018855,1,False,shock septico [SEP] falla multiple de organos ...


,id_caso,clase_predicha,clase_real,score_etapa1_hacia_mencion,score_etapa2_hacia_clase_2,abs_score_etapa2,pred_etapa1_cruda,reasignado_de_0_a_2,texto
1,CASO_002951,2,1,2.213094,0.612212,0.612212,1,False,tumor d comportamiento incierto o deconocido d...
86,CASO_002865,2,1,1.616272,0.104461,0.104461,1,False,neoplasia abdominal maligna
398,CASO_002511,2,1,0.984561,0.336357,0.336357,1,False,disfunci”n organica [SEP] sepsis de origen abd...
476,CASO_002202,2,1,1.337647,0.266994,0.266994,1,False,tumor maligno de las vias biliares [SEP] diabe...
704,CASO_000990,2,1,1.006016,0.172760,0.172760,1,False,enfermedad cerebrovascular isquemica [SEP] hip...
1028,CASO_003363,0,1,-0.637667,-0.315092,0.315092,0,False,choque cardiogenico [SEP] infarto agudo del mi...
1412,CASO_003261,2,1,1.535257,0.381371,0.381371,1,False,hipoxemia refractaria [SEP] síndrome de distre...
1441,CASO_003369,2,1,1.088699,0.097150,0.097150,1,False,cáncer pulmonar [SEP] tabaquismo [SEP] enferme...
1485,CASO_001695,2,1,1.133257,0.333004,0.333004,1,False,paro cardiaco [SEP] tumor de comportamiento in...
1966,CASO_000445,2,1,0.921130,0.018855,0.018855,1,False,shock septico [SEP] falla multiple de organos ...



--- Definición de Columnas ---
id_caso: Identificador único del caso.
clase_predicha: Clase predicha por el modelo final (0: sin mención de cáncer, 1: mención clara, 2: mención probable/sospechosa).
clase_real: Clase real o etiquetada del caso (si disponible).
score_etapa1_hacia_mencion: Score de la primera etapa del modelo (0 vs 1+2). Positivo indica mención, negativo sin mención.
score_etapa2_hacia_clase_2: Score de la segunda etapa del modelo (1 vs 2). Positivo indica clase 2, negativo clase 1.
abs_score_etapa2: Valor absoluto del score de la segunda etapa. Valores bajos indican baja confianza.
pred_etapa1_cruda: Predicción original de la primera etapa antes de cualquier reasignación (0 o 1).
reasignado_de_0_a_2: Booleano que indica si el caso fue reasignado de clase 0 a clase 2 debido a una baja confianza en la etapa 1.
texto: El texto combinado utilizado para la clasificación.
Archivo generado: resultados_train_svm_solo_errores.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>